In [2]:
import numpy as np

def triangle_B_area(coords):
    """coords: array 3x2 con (x,y) de cada nodo"""
    x1,y1 = coords[0]
    x2,y2 = coords[1]
    x3,y3 = coords[2]
    
    area = 0.5 * abs(x1*(y2-y3) + x2*(y3-y1) + x3*(y1-y2))
    
    b1, b2, b3 = y2-y3, y3-y1, y1-y2
    c1, c2, c3 = x3-x2, x1-x3, x2-x1
    
    B = (1/(2*area)) * np.array([
        [b1, 0,  b2, 0,  b3, 0 ],
        [0,  c1, 0,  c2, 0,  c3],
        [c1, b1, c2, b2, c3, b3]
    ])
    return B, area

def plane_strain_D(E, nu):
    factor = E / ((1+nu)*(1-2*nu))
    D = factor * np.array([
        [1-nu,   nu,     0],
        [nu,     1-nu,   0],
        [0,      0,      (1-2*nu)/2]
    ])
    return D

def triangle_stiffness(coords, E, nu, thickness=1.0):
    B, area = triangle_B_area(coords)
    D = plane_strain_D(E, nu)
    K = B.T @ D @ B * area * thickness
    return K

# ---- Ejemplo: triángulo simple ----
coords = np.array([
    [0, 0],
    [1, 0],
    [0, 1]
])

E = 200e9
nu = 0.29

K = triangle_stiffness(coords, E, nu)
print(K.shape)   # Debe ser (6,6)
print(K)

# Vector de fuerzas global (6 valores: fx1,fy1,fx2,fy2,fx3,fy3)
F = np.array([0, 0, 5000, 0, 0, 0])

# DOF fijos: u1(idx0), v1(idx1), u3(idx4)
# DOF libres: v3(idx2 es u2, idx3 es v2), y v3(idx5)... 

fixed_dofs = [0, 1, 4]           # u1, v1, u3
free_dofs = [2, 3, 5]            # u2, v2, v3

K_free = K[np.ix_(free_dofs, free_dofs)]
F_free = F[free_dofs]

u_free = np.linalg.solve(K_free, F_free)
print(u_free)

(6, 6)
[[ 1.69804356e+11  9.22849760e+10 -1.31044666e+11 -3.87596899e+10
  -3.87596899e+10 -5.35252861e+10]
 [ 9.22849760e+10  1.69804356e+11 -5.35252861e+10 -3.87596899e+10
  -3.87596899e+10 -1.31044666e+11]
 [-1.31044666e+11 -5.35252861e+10  1.31044666e+11  0.00000000e+00
   0.00000000e+00  5.35252861e+10]
 [-3.87596899e+10 -3.87596899e+10  0.00000000e+00  3.87596899e+10
   3.87596899e+10  0.00000000e+00]
 [-3.87596899e+10 -3.87596899e+10  0.00000000e+00  3.87596899e+10
   3.87596899e+10  0.00000000e+00]
 [-5.35252861e+10 -1.31044666e+11  5.35252861e+10  0.00000000e+00
   0.00000000e+00  1.31044666e+11]]
[ 4.5795e-08  0.0000e+00 -1.8705e-08]
